<a href="https://colab.research.google.com/github/rtovardev/taller-langchain-agentes/blob/main/notebook/taller_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Construye tu primer agente de IA con LangChain

**Taller práctico · EurusConf 2026**

Este notebook es la **Parte 1** del taller y corre en **Google Colab** (no instalas nada en tu máquina).

Vas a construir un **agente investigador** y luego lo vas a hacer **sólido**: que se vea bien (streaming), que recuerde (memoria) y que devuelva datos limpios (salida estructurada).

> En la **Parte 2** pasamos a VS Code y llevamos este mismo agente a un **proyecto real** (con `uv` + `src/`). Eso vive en el repo.

**Cómo correrlo:** ve celda por celda (Shift+Enter). Corre primero el Setup.

## 0. Setup

In [44]:
# Instala las librerías (en Colab). Tarda ~1 min.
%pip install langchain langchain-google-genai langchain-openai langchain-community ddgs python-dotenv pydantic requests tavily

  Preparing metadata (setup.py) ... done
  Created wheel for tavily: filename=tavily-1.1.0-py3-none-any.whl size=6128 sha256=edd0e52cac5782d2b76d409f71e1dff0d7e9f00c55cdecc48fa7624a18bb8377
  Stored in directory: /root/.cache/pip/wheels/a7/67/b7/9aec4851724de28ac2bc34ff10b042af43d7f2dd1552e5906e
Successfully built tavily


In [38]:
from google.colab import userdata

# Cargando las credenciales desde las variables de entorno
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [39]:
from langchain.chat_models import init_chat_model
from IPython.display import display, Markdown

def consultar_modelo(api_key: str, query: str) -> str:
    """
    Recibe una API Key de OpenRouter y un query, y devuelve la respuesta del modelo.
    """
    model = init_chat_model(
        model="openai/gpt-4o-mini",
        model_provider="openai",
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key
    )

    respuesta = model.invoke(query)

    return respuesta.content

res = consultar_modelo(OPENROUTER_API_KEY, "¿Qué es un agente de IA?")

display(Markdown(res))

Un agente de IA (Inteligencia Artificial) es un sistema o programa que utiliza técnicas y algoritmos de inteligencia artificial para realizar tareas de manera autónoma o semi-autónoma. Estos agentes pueden interactuar con su entorno, procesar información, tomar decisiones y resolver problemas basándose en los datos que reciben.

Existen diferentes tipos de agentes de IA, que pueden clasificarse según su nivel de autonomía y capacidad para aprender y adaptarse. Algunos ejemplos incluyen:

1. **Agentes reactivos**: Operan basándose en datos en tiempo real y responden a estímulos inmediatos sin un modelo interno del mundo.

2. **Agentes basados en objetivos**: Tienen objetivos específicos que buscan alcanzar y pueden planificar acciones en función de esos objetivos.

3. **Agentes con aprendizaje**: Utilizan técnicas de aprendizaje automático para mejorar su desempeño con el tiempo al aprender de la experiencia y los datos.

4. **Agentes inteligentes**: Integran múltiples capacidades, como la percepción, el razonamiento y el aprendizaje, para tomar decisiones complejas.

Los agentes de IA se utilizan en una variedad de aplicaciones, como asistentes virtuales (como Siri o Alexa), chatbots, sistemas de recomendación, vehículos autónomos, y más. Su objetivo es automatizar tareas, mejorar la eficiencia y ofrecer soluciones a problemas que tradicionalmente requerían intervención humana.

## Recordatorio rápido

Ya vimos la teoría en las diapositivas. Lo esencial para programar:

- Un **agente** = modelo + herramientas + un *loop*: recibe una tarea, decide si usar una herramienta, observa el resultado y repite hasta responder.
- Hoy lo armamos con **`create_agent`** (de LangChain), que por debajo corre sobre **LangGraph**.
- Una **herramienta (`@tool`)** es solo una función de Python con una descripción. El modelo decide cuándo llamarla.

## Actividad 1 — Tu agente investigador

Primero un agente con una herramienta trivial (para ver el *loop*), y luego el de verdad: uno que **busca en la web**.

In [40]:
# Definimos el modelo que vamos a utilizar para la creacion de agentes — Facilitador (Modelo Pago)
model_facilitador = init_chat_model(
    model="openai/gpt-4.1-mini",
    model_provider="openai",
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

# Definimos el modelo que vamos a utilizar para la creacion de agentes — Estudiante (Modelo Gratuito)
model_estudiante = init_chat_model(
    model="gemini-2.5-flash",
    model_provider="google_genai",
    api_key=GEMINI_API_KEY
)

### ⚠️ DISCLAIMER: Límites de la API Gratuita

Para este taller usaremos la capa gratuita de **Google Gemini**, la cual está limitada estrictamente a:
* **15 peticiones por minuto (RPM)** (El tope crítico).
* **Hasta 500-1500 peticiones por día (RPD)**, dependiendo del modelo [Google AI Studio](https://ai.google.dev/gemini-api/docs/rate-limits).

> **La trampa del Agente:** Un solo "Play" ejecuta entre 2 y 4 peticiones internas. Si spameas el botón tras un error, consumirás tus 15 peticiones en segundos y Google te bloqueará por un minuto *(Error 429)*. Revisa tu código antes de reintentar.

In [62]:
# DEMO — calentamiento: un agente con UNA herramienta simple, para ver el loop
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from datetime import datetime

@tool
def hora_actual(zona: str = "local") -> str:
    """Devuelve la fecha y hora actual."""
    return datetime.now().strftime("%Y-%m-%d %H:%M")

agente = create_agent(
    model=model_facilitador,
    tools=[hora_actual],
    system_prompt="Eres un asistente útil. Usa las herramientas cuando ayuden.",
)

r = agente.invoke({"messages": [{"role": "user", "content": "¿Qué fecha y hora es?"}]})
print(r["messages"][-1].content)

La fecha y hora actuales son 2026-06-26 04:07.


### Tu turno
Enfoca el investigador a **TU** tema (tu carrera, un hobby) cambiando el `system_prompt`, y hazle una pregunta actual.

In [45]:
# Importando modulos necesarios
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from tavily import TavilyClient


# TODO: crea tu investigador con un system_prompt enfocado a tu tema
print("Buena suerte!")

Buena suerte!


### Solución A1

In [63]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from tavily import TavilyClient
from IPython.display import display, Markdown
from google.colab import userdata
from datetime import datetime

# Inicializamos el objeto de la clase TavilyClient
client = TavilyClient(api_key=userdata.get('TAVILY_API_KEY'))

@tool
def buscar(query: str) -> str:
  """Busca información en tiempo real en internet sobre un tema específico."""
  response = client.search(
    query,
    max_results=3,
  )
  return response['results']

mi_investigador = create_agent(
    model=model_estudiante,
    tools=[buscar],
    system_prompt=f"Eres un investigador experto en tecnología. Responde claro y con datos actuales. Fecha actual: {datetime.now().strftime("%Y-%m-%d %H:%M")}",
)

r = mi_investigador.invoke({"messages": [{"role": "user", "content": "¿Qué novedades hay en inteligencia artificial?"}]})

# Extraemos el contenido asegurándonos de que sea un string para Markdown
contenido_crudo = r["messages"][-1].content

if isinstance(contenido_crudo, list):
    # Si es una lista (formato multi-modal), extraemos solo el texto
    texto_final = "".join([m.get("text", "") for m in contenido_crudo if isinstance(m, dict)])
else:
    texto_final = contenido_crudo

display(Markdown(texto_final))

La inteligencia artificial está experimentando un desarrollo vertiginoso en diversos frentes. Aquí te presento las novedades más destacadas, con datos actualizados a junio de 2026:

**1. Avances en Modelos y Hardware:**

*   **Nuevos Chips y Modelos:** OpenAI ha lanzado su primer chip, "Jalapeño", buscando reducir su dependencia de Nvidia. Por su parte, Anthropic continúa mejorando su IA Claude, que también está expandiendo sus herramientas al sector legal. Google sigue apostando fuerte con Gemini y Android Intelligence, integrando capacidades de IA directamente en el sistema operativo Android.
*   **Supercomputación:** China alberga el supercomputador más potente del mundo, lo que impulsa la investigación y el desarrollo en IA a gran escala.

**2. Aplicaciones y Casos de Uso Innovadores:**

*   **Productividad y Asistencia:** Se están viendo dispositivos "hipervitaminados" con IA y Android que prometen revolucionar la toma de apuntes en clase. Claude Tag, por ejemplo, convierte a Claude en un compañero en Slack para mejorar la colaboración.
*   **Entretenimiento y Consumo:** Samsung está integrando IA en sus televisores para el Mundial, prometiendo mejoras en la experiencia visual.
*   **Robótica Avanzada:** Unitree ha presentado un robot de casi 500 kilos, el GD01, que se asemeja a una "montura mecha". Además, robots de la compañía Figure ya están trabajando durante días, mostrando avances significativos en robótica autónoma.
*   **Sectores Estratégicos:** Airbus está probando la IA en una de las maniobras más críticas de un vuelo: el aterrizaje, lo que podría aumentar la seguridad y eficiencia. La IA también está siendo utilizada para recuperar activos perdidos, como el caso de Claude ayudando a recuperar bitcoins.
*   **Finanzas Personales:** ChatGPT de OpenAI está incorporando nuevas herramientas para la gestión de finanzas personales.

**3. Panorama Empresarial y Desafíos:**

*   **Éxito Financiero:** Algunas empresas están logrando monetizar la IA de manera efectiva, generando nuevas fortunas en el sector. Adobe, por ejemplo, está teniendo un buen desempeño, aunque el futuro a largo plazo aún es incierto.
*   **Competencia y Litigios:** Anthropic ha acusado a Alibaba de "destilar" su IA Claude, lo que subraya la creciente competencia y los desafíos legales en torno a la propiedad intelectual de la IA. También continúan los debates y juicios, como el de Elon Musk contra OpenAI.
*   **Adopción y Reacción Social:** Mientras Malta ha sellado un acuerdo con OpenAI para ofrecer acceso a ChatGPT Plus a sus ciudadanos, también se observa un creciente rechazo a la IA entre los estudiantes, lo que plantea interrogantes sobre la integración de estas tecnologías en la educación.
*   **Ciberseguridad:** Nuevas soluciones como Daybreak y Mythos están surgiendo para abordar los desafíos de ciberseguridad en el ámbito de la IA.

En resumen, la IA sigue expandiéndose en sus capacidades y aplicaciones, redefiniendo industrias y generando un intenso debate sobre su futuro, implicaciones éticas y el impacto en la sociedad.

## Actividad 2 — Hazlo sólido

El mismo agente, pero mejor: **streaming** (se ve la respuesta llegar), **memoria** (recuerda la conversación) y **salida estructurada** (devuelve datos limpios, no texto suelto).

### 2.1 Streaming — que se vea la respuesta llegar en vivo

In [64]:
# En vez de esperar toda la respuesta, la imprimimos token por token (FALTA)

[{'type': 'text', 'text': 'Los agentes de IA son cruciales porque pueden percibir su entorno, tomar decisiones y actuar de forma autónoma para lograr objetivos específicos, incluso en entornos complejos y dinámicos. Su importancia radica en la', 'extras': {'signature': 'CnQBDDnWxwDwlOfLtkB8RIaX62scxjUZoKe7nRf1b4xBZcu945Yi/JjL+HRzEkNFKKo2wVgZ5tKI2CVfdLQoqe76evEeLhhHY4kaVyCBMQOQ2PGlSZK5JHypgZ0oS2PD9RSyvsFNVSKos6AvxmPgJkCxAchnSQr8AQEMOdbHOSOPQBpB7MPVY+ANXfSyA43pnMrj0CNwpY85A4Gj+n/SaO6Rex89MSYEGL2Nkvpk4Ci1tFjBbzTVkSz5qJ/A91awZf2CziCf3+/F8UgjIdt3cpaXJ6jWoL/oQBSLKUqG2IbsyLXKgH+2fpud42aeU9QSVv60M65kq+zo5Klmeumzw7mVc+uA9k3gfXVCC41T2lWKJZtTnhyblbS/WDyspOgcO9o5p9t9K3JmLjiOyi+UGFLckWZ2yByZ5J/TfTgliwMCA9bBVaOJbBC+8QNcA/+2ZtVchunNndZQWwAEffikhlek08KI1t8nJRAGs5P/afYoqbpCugpBAQw51se/dWdxmR/PWE8mXNrObj7tc7JNXtB64F75h388mZqRERqunqMqbEAJ688ALdY2pnVn+Hn0mU7znHCJt20='}, 'index': 0}] capacidad de automatizar tareas complejas, optimizar procesos y resolver problemas que requieren razonamiento y adaptación. 

### 2.2 Memoria — que recuerde la conversación
Con un `checkpointer` y un `thread_id`, el agente recuerda lo que ya hablaron.

In [65]:
from langgraph.checkpoint.memory import InMemorySaver

investigador_memoria = create_agent(
    model=model_facilitador,
    tools=[buscar],
    system_prompt="Eres un investigador que recuerda la conversación.",
    checkpointer=InMemorySaver(),
)
config = {"configurable": {"thread_id": "demo-1"}}
investigador_memoria.invoke({"messages": [{"role": "user", "content": "Me interesan los agentes de IA."}]}, config)
r = investigador_memoria.invoke({"messages": [{"role": "user", "content": "¿Qué tema te dije que me interesa?"}]}, config)
print(r["messages"][-1].content)  # debería recordarlo

Me dijiste que te interesan los agentes de IA. ¿Quieres que te proporcione más información sobre ellos?


### 2.3 Salida estructurada — datos limpios en vez de texto suelto
Útil para integrar con otros sistemas (guardar en una base, mandar a una API...).

In [66]:
from pydantic import BaseModel, Field

class Resumen(BaseModel):
    tema: str = Field(description="el tema consultado")
    puntos_clave: list[str] = Field(description="3 a 5 puntos clave, en frases cortas")

estructurado = model.with_structured_output(Resumen)
print(estructurado.invoke("Resume en puntos clave qué es LangChain"))

tema='LangChain' puntos_clave=['Es un framework para construir aplicaciones con modelos de lenguaje grande (LLMs).', 'Facilita la integración de LLMs con datos externos y APIs.', 'Permite encadenar múltiples llamadas a modelos para tareas complejas.', 'Se utiliza para crear agentes, chatbots y sistemas de recuperación de información.', 'Ofrece componentes modulares como promts, cadenas y gestores de memoria.']


### Tu turno
Dale **memoria** a tu investigador (con tu propio `thread_id`) y compruébalo en dos turnos.

In [ ]:
# TODO: crea tu investigador con checkpointer=InMemorySaver(), define un config con tu thread_id,
# invócalo dos veces y comprueba que recuerda algo del primer turno.

### Solución A2

In [ ]:
mi_agente = create_agent(
    model=model,
    tools=[buscar_web],
    system_prompt="Eres mi asistente de investigación. Recuerda lo que te digo.",
    checkpointer=InMemorySaver(),
)
cfg = {"configurable": {"thread_id": "mi-sesion"}}
mi_agente.invoke({"messages": [{"role": "user", "content": "Estoy investigando sobre energías renovables."}]}, cfg)
r = mi_agente.invoke({"messages": [{"role": "user", "content": "Dame una noticia reciente sobre lo que estoy investigando."}]}, cfg)
print(r["messages"][-1].content)

## ¿Por qué construir tu propio agente si Claude Code ya existe?

Para tareas generales, usa lo que ya existe. Construyes el **tuyo** cuando tienes un flujo específico y repetible que se beneficia de:

- **Control:** tú decides qué herramientas tiene y qué puede hacer.
- **Integración:** se conecta a **tus** sistemas (tu base de datos, tu API, tu Notion).
- **Costo y escala:** corre desatendido con el modelo más barato que sirva.
- **Automatización real:** se dispara por eventos, no es un chat.

## Parte 2 — del notebook al proyecto real (en VS Code)

Hasta aquí, todo en el notebook. Ahora llevamos **este mismo agente** a un **proyecto real** con la estructura que usa LangChain:

```
src/agente/
  agent.py      # create_agent(model, tools, system_prompt, checkpointer)
  tools.py      # las @tool
  prompts.py    # el system prompt
main.py         # punto de entrada (agent.invoke)
.env  ·  pyproject.toml  ·  uv.lock
```

Lo hacemos juntos en el repo con **`uv`** (el gestor de paquetes moderno). El paso a paso está en el **README**.

### Si te sobra tiempo: 3 ideas para tu propio agente
Construye uno tuyo en ese proyecto, conectado a algo real (todas con token, sin OAuth):

1. **Investigador + scraping** — busca y además extrae info de una página.
2. **Tu productividad** — Notion o Todoist: convierte lo que escribes en tareas/notas.
3. **Datos en vivo** — una API pública (clima, finanzas, noticias) + memoria.

El código de ejemplo de cada idea está en el **README** del repo.

---

**Recursos:** https://docs.langchain.com · **MART Automations:** https://martautomations.com